# Azure VM workload: exploratory analysis

**Question:** how predictable is this cloud workload, and what should an infrastructure team do with a forecast?

This notebook is descriptive. It answers assignment part A (understand the workload) and, in the last section, reads the
pipeline outputs to analyse forecast errors (part F). It never fits or tunes a model.

**Leakage rule.** Everything that could influence modelling decisions (thresholds, peak definitions, volatility estimates) uses
only the first 110 observations (`train`). Full-sample numbers are marked *descriptive*. The 24 holdout observations are only used
in the final section, after the forecast was frozen, and only for reporting.

Run the pipeline first (`uv run workload-forecasting predict`) so the last section can find its inputs.
Sections: 1 data quality, 2 lifetimes and churn, 3 CPU distributions, 4 temporal patterns, 5 segments, 6 peaks and lows,
7 why the aggregate is smooth, 8 forecast diagnostics, 9 summary export.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display as ipy_display
from statsmodels.tsa.stattools import acf, adfuller

from workload_forecasting.data.aggregation import aggregate_cpu_data
from workload_forecasting.data.loader import load_datasets

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA, FIG, TRAIN_DIR = (
    ROOT / "challenge_data",
    ROOT / "reports" / "figures",
    ROOT / "reports" / "training",
)
STEP, H = 300, 24  # seconds per observation, holdout length (must match params.yaml)
FIG.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "savefig.bbox": "tight"})


def save(name):
    """Save the current figure to the reports/figures directory and display it in the notebook."""
    plt.savefig(FIG / f"{name}.png")
    plt.show()


def _key(v):
    """Sort bucket labels numerically when possible ('2' < '4' < '24' < '>24' ...)."""
    n = pd.to_numeric(pd.Series([v]), errors="coerce").iloc[0]
    return (bool(np.isnan(n)), 0.0 if np.isnan(n) else float(n), str(v))

In [ ]:
cpu, meta = load_datasets(DATA)
y = aggregate_cpu_data(cpu, STEP)  # same function the pipeline uses
train, test = y.iloc[:-H].reset_index(drop=True), y.iloc[-H:].reset_index(drop=True)
T0, T_END, CUT = cpu["timestamp"].min(), cpu["timestamp"].max(), test["timestamp"].iloc[0]


def hours(ts):
    """Convert a timestamp to hours since the start of the dataset."""
    return (ts - T0) / 3600


print(
    f"{len(cpu):,} readings | {cpu.vm_id.nunique():,} VMs | train {len(train)} steps | holdout {len(test)} steps (starts at {hours(CUT):.2f} h)"
)

## 1. Cadence, coverage, missingness, duplicates

The data card promises a complete rectangular panel. We verify rather than assume, because every later conclusion
(for example "the VM count is constant") depends on it.

In [ ]:
srt = cpu.sort_values(["vm_id", "timestamp"])
gaps = srt.groupby("vm_id")["timestamp"].diff().dropna()
per_vm = cpu.groupby("vm_id").size()
per_ts = cpu.groupby("timestamp")["vm_id"].nunique()
cols = ["min_cpu", "max_cpu", "avg_cpu"]

max_perc = 100
quality = pd.Series(
    {
        "readings": len(cpu),
        "VMs": cpu["vm_id"].nunique(),
        "metadata rows": len(meta),
        "VMs without metadata": len(set(cpu["vm_id"]) - set(meta["vm_id"])),
        "distinct timestamps": cpu["timestamp"].nunique(),
        "span (hours)": round((T_END - T0) / 3600 + STEP / 3600, 2),
        "readings per VM (min)": per_vm.min(),
        "readings per VM (max)": per_vm.max(),
        "VMs per timestamp (min)": per_ts.min(),
        "VMs per timestamp (max)": per_ts.max(),
        "duplicate (vm_id, timestamp)": int(cpu.duplicated(["vm_id", "timestamp"]).sum()),
        "null cells": int(cpu.isna().sum().sum()),
        "gaps != 300 s": int((gaps != STEP).sum()),  # ty: ignore[invalid-argument-type]
        "avg_cpu outside [min, max]": int(
            ((cpu.avg_cpu < cpu.min_cpu - 1e-6) | (cpu.avg_cpu > cpu.max_cpu + 1e-6)).sum()
        ),
        "readings outside [0, 100]": int(
            ((cpu[cols] < 0) | (cpu[cols] > max_perc)).any(axis=1).sum()
        ),
    }
).to_frame("value")
ipy_display(quality)

fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
gaps.value_counts().sort_index().plot.bar(ax=ax[0], logy=True)
ax[0].set(title="(a) Gap between consecutive readings", xlabel="seconds", ylabel="count (log)")
ax[1].plot(hours(per_ts.index), per_ts.values)
ax[1].axvline(hours(CUT), color="red", ls="--", label="holdout starts")
ax[1].set(
    title="(b) VMs reporting per timestamp",
    xlabel="hours since first observation",
    ylim=(0, per_ts.max() * 1.1),
)
ax[1].legend()
ax[2].hist(per_vm, bins=np.arange(per_vm.min().item() - 1.5, per_vm.max().item() + 2.5, 1))
ax[2].set(title="(c) Readings per VM", xlabel="readings", ylabel="VMs")
save("eda_01_cadence_coverage")

**Operational reading.** A single 300 s cadence, no gaps, no duplicates, no nulls, and every VM present at every timestamp means no
resampling or imputation is needed. The flip side: because only VMs with complete traces were kept, `active_vm_count`
is constant **by construction**, not because the fleet is stable. The dataset cannot tell us anything about arrival and departure rates.

## 2. VM lifetimes and churn

Reading data cannot show churn here, but the metadata creation and deletion timestamps can show how the sampled VMs relate to the window.

In [ ]:
m = meta.copy()
m["life_h"] = (m["timestamp_vm_deleted"] - m["timestamp_vm_created"]) / 3600
pre, survives = m["timestamp_vm_created"] <= T0, m["timestamp_vm_deleted"] >= T_END
m["status"] = np.select(
    [pre & survives, pre & ~survives, ~pre & survives, ~pre & ~survives],
    [
        "existing before window, survives",
        "existing before window, deleted in window",
        "created in window, survives",
        "created and deleted in window",
    ],
    default="unknown",
)

status = m["status"].value_counts()
ipy_display(status.to_frame("VMs"))

life = m["life_h"].clip(lower=0.1)
min_h, max_h = 24, 168
print(
    f"median lifetime {life.median():,.0f} h | share >= 24 h: {(m.life_h >= min_h).mean():.1%} | share >= 168 h (1 week): {(m.life_h >= max_h).mean():.1%}"
)
ipy_display(m.groupby("vm_category")["life_h"].describe(percentiles=[0.5]).round(0))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
status.reindex(
    [
        "existing before window, survives",
        "existing before window, deleted in window",
        "created in window, survives",
        "created and deleted in window",
    ]
).fillna(0).plot.barh(ax=ax[0])
ax[0].set(title="(a) Lifetime status vs observation window", xlabel="VMs")
ax[1].hist(life, bins=np.logspace(np.log10(life.min()), np.log10(life.max()), 40))
ax[1].axvline((T_END - T0) / 3600, color="red", ls="--", label="observation window")
ax[1].set(
    xscale="log", title="(b) VM lifetime (created to deleted)", xlabel="hours (log)", ylabel="VMs"
)
ax[1].legend()
save("eda_02_lifetimes_churn")

**Operational reading.** If virtually all VMs existed before the window and survive it, the sample is a **survivor population**: long-lived,
steady tenants. It is a good basis for a short-horizon level forecast, but it understates the demand volatility a real fleet gets from
short-lived VMs. A capacity policy built on it should be treated as a floor for headroom, not a ceiling.

## 3. CPU utilisation distributions

Reading-level and VM-level views answer different questions. The reading level describes instantaneous load; the VM level shows how the
fleet average is made up.

In [ ]:
vm_mean = cpu.groupby("vm_id")["avg_cpu"].mean()
qs = (
    cpu["avg_cpu"]
    .quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
    .rename("avg_cpu reading quantile")
)
ipy_display(qs.to_frame().T.round(2))

ranked = vm_mean.sort_values(ascending=False)
cum = ranked.cumsum() / ranked.sum()
top = {p: float(cum.iloc[max(int(len(cum) * p) - 1, 0)]) for p in (0.01, 0.05, 0.10, 0.20)}

min_prc, max_prc, max_cpu_prc = 5, 50, 90
stats = {
    "mean avg_cpu (all readings)": cpu["avg_cpu"].mean(),
    "median VM mean": vm_mean.median(),
    "VMs with mean < 5 %": (vm_mean < min_prc).mean(),
    "VMs with mean > 50 %": (vm_mean > max_prc).mean(),
    "readings with max_cpu >= 90 %": (cpu["max_cpu"] >= max_cpu_prc).mean(),
    **{f"share of total CPU from top {int(p * 100)}% VMs": v for p, v in top.items()},
}
ipy_display(pd.Series(stats).round(4).to_frame("value"))

fig, ax = plt.subplots(2, 2, figsize=(13, 7))
ax[0, 0].hist(cpu["avg_cpu"], bins=100, log=True)
ax[0, 0].set(title="(a) avg_cpu, all readings", xlabel="%", ylabel="readings (log)")
ax[0, 1].hist(vm_mean, bins=100)
ax[0, 1].set(title="(b) per-VM mean avg_cpu", xlabel="%", ylabel="VMs")
ax[1, 0].plot(np.sort(vm_mean.values), np.arange(1, len(vm_mean) + 1) / len(vm_mean))
ax[1, 0].set(title="(c) CDF of per-VM mean", xlabel="%", ylabel="fraction of VMs")
ax[1, 1].plot(np.arange(1, len(cum) + 1) / len(cum), cum.values)
ax[1, 1].plot([0, 1], [0, 1], "k:", lw=1)
ax[1, 1].set(
    title="(d) CPU concentration",
    xlabel="fraction of VMs (heaviest first)",
    ylabel="cumulative share of CPU",
)
fig.tight_layout()
save("eda_03_cpu_distributions")

**Operational reading.** A strongly right-skewed VM distribution (many near-idle VMs, a small heavy tail) has two consequences.
First, the fleet mean is low, so the question for capacity is rarely "will we run out?" and more often "how much idle capacity can we
reclaim safely?". Second, the mean is carried by relatively few VMs, so segment analysis (section 5) matters more than the
single aggregate.

## 4. Temporal patterns of the targets

The span is only about 11 hours, so a daily or weekly cycle cannot be estimated. The trace clock is relative to the source trace,
so we do not claim any time-of-day effect. We look at level, autocorrelation, stationarity and how quickly persistence error grows with horizon.

In [ ]:
col = "aggregate_cpu_utilization"
s_tr = train[col].to_numpy()

fig, ax = plt.subplots(2, 1, figsize=(12, 6.5), sharex=True)
ax[0].plot(hours(y.timestamp), y[col], label="aggregate CPU %")
ax[0].plot(
    hours(y.timestamp), y[col].rolling(12, min_periods=1).mean(), label="1 h rolling mean", lw=2
)
ax[0].axvspan(
    hours(CUT),
    hours(T_END) + STEP / 3600,
    color="red",
    alpha=0.08,
    label="holdout (not used for any decision)",
)
ax[0].set(title="aggregate_cpu_utilization", ylabel="%")
ax[0].legend()
ax[1].plot(hours(y.timestamp), y["active_vm_count"])
ax[1].axvspan(hours(CUT), hours(T_END) + STEP / 3600, color="red", alpha=0.08)
ax[1].set(
    title="active_vm_count",
    xlabel="hours since first observation",
    ylabel="VMs",
    ylim=(0, y["active_vm_count"].max() * 1.1),
)
save("eda_04_targets")

lags = 36
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
for a, series, name in ((ax[0], s_tr, "level"), (ax[1], np.diff(s_tr), "first difference")):
    a.bar(range(lags + 1), acf(series, nlags=lags), width=0.8)
    a.axhline(1.96 / np.sqrt(len(series)), color="red", ls="--")
    a.axhline(-1.96 / np.sqrt(len(series)), color="red", ls="--")
    a.set(title=f"ACF of {name} (train)", xlabel="lag (5-min steps)")
save("eda_05_acf")

adf_level, adf_diff = adfuller(s_tr)[1], adfuller(np.diff(s_tr))[1]
print(f"ADF p-value: level {adf_level:.3f} | first difference {adf_diff:.3g}")
print(
    f"lag-1 autocorrelation {acf(s_tr, nlags=1)[1]:.3f} | lag-12 {acf(s_tr, nlags=12)[12]:.3f} | lag-24 {acf(s_tr, nlags=24)[24]:.3f}"
)
print(
    f"train range {s_tr.min():.2f}-{s_tr.max():.2f} % | mean {s_tr.mean():.3f} | std {s_tr.std(ddof=1):.3f}"
)

In [ ]:
# Persistence error vs horizon, against the error of always predicting the training mean (train only).
mean_err = np.abs(s_tr - s_tr.mean()).mean()
rows = [
    {
        "horizon_steps": h,
        "horizon_min": h * 5,
        "persistence_MAE": np.abs(s_tr[h:] - s_tr[:-h]).mean(),
        "train_mean_MAE": mean_err,
    }
    for h in (1, 3, 6, 12, 18, 24)
]
horizon = pd.DataFrame(rows).set_index("horizon_steps")
horizon["persistence_better"] = horizon["persistence_MAE"] < horizon["train_mean_MAE"]
ipy_display(horizon.round(3))

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(horizon.index, horizon["persistence_MAE"], marker="o", label="persistence (last value)")
ax.axhline(mean_err, color="C1", ls="--", label="always the training mean")
ax.set(
    title="Where does persistence stop beating the mean? (train only)",
    xlabel="horizon (steps of 5 min)",
    ylabel="MAE (% points)",
)
ax.legend()
save("eda_06_persistence_vs_horizon")

**Operational reading.** The aggregate is a slowly moving level with short-lived noise. If persistence error grows toward (or past) the
error of simply predicting the training mean at the 2-hour horizon, then a pure last-value forecast over-trusts the latest point.
A blend of "last value" and "long-run mean" is the natural cheap remedy. This is the evidence behind the ensemble candidate the
cross-validation later selects. The trace is too short to support seasonal or trend models, which is why cross-validation also does not
reward them.

In [ ]:
y["hour"] = ((y["timestamp"] - T0) // 3600).astype(int)
tr = y.iloc[: len(train)]
hourly = tr.groupby("hour")[col].agg(["mean", "min", "max", "count"]).round(3)
ipy_display(hourly)

hi_the = tr[col].quantile(0.9)
lo_the = tr[col].quantile(0.1)
peaks = tr[tr[col] >= hi_the][["timestamp", col]].assign(
    hours_since_start=lambda d: hours(d.timestamp).round(2)
)
lows = tr[tr[col] <= lo_the][["timestamp", col]].assign(
    hours_since_start=lambda d: hours(d.timestamp).round(2)
)
print(f"peak threshold (train p90) = {hi_the:.3f} %, low threshold (train p10) = {lo_the:.3f} %")
print("peak steps cluster at hours:", sorted(set(np.floor(peaks.hours_since_start).astype(int))))
print("low steps cluster at hours:", sorted(set(np.floor(lows.hours_since_start).astype(int))))

## 5. Workload differences across VM segments

Segments (category, core bucket, memory bucket) come from metadata, which is fine for *describing* the workload.
Lifetime statistics in the metadata (`avg_cpu`, `max_cpu`, `p95_max_cpu`, deletion time) cover periods that include the holdout and beyond,
so they are **not** used as forecasting features.

In [ ]:
seg = cpu.merge(
    meta[["vm_id", "vm_category", "virtual_core_bucket", "memory_gb_bucket"]], on="vm_id"
)
for c in ("vm_category", "virtual_core_bucket", "memory_gb_bucket"):
    seg[c] = seg[c].astype(str)


def seg_table(c):
    """Compute descriptive statistics for a given categorical column."""
    g = seg.groupby(c)
    vm_level = seg.groupby([c, "vm_id"])["avg_cpu"].mean()
    t = pd.DataFrame(
        {
            "VMs": g["vm_id"].nunique(),
            "mean_avg_cpu": g["avg_cpu"].mean(),
            "p95_of_vm_means": vm_level.groupby(c).quantile(0.95),
            "mean_max_cpu": g["max_cpu"].mean(),
        }
    )
    t["vm_share"] = t["VMs"] / t["VMs"].sum()
    t["cpu_share"] = (t["VMs"] * t["mean_avg_cpu"]) / (t["VMs"] * t["mean_avg_cpu"]).sum()
    return t.loc[sorted(t.index, key=_key)]


tables = {c: seg_table(c) for c in ("vm_category", "virtual_core_bucket", "memory_gb_bucket")}
for c, t in tables.items():
    print(f"\n{c} (descriptive, full sample)")
    ipy_display(t.round(3))

In [ ]:
train_seg = seg[seg["timestamp"] < CUT]  # everything below is train-only
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
max_perc = 100
for a, c in zip(ax, tables, strict=True):
    big = tables[c].index[tables[c]["VMs"] >= max_perc]
    ts = train_seg[train_seg[c].isin(big)].groupby(["timestamp", c])["avg_cpu"].mean().unstack()
    for k in sorted(ts.columns, key=_key):
        a.plot(hours(ts.index), ts[k], label=k)
    a.set(title=f"mean avg_cpu by {c} (train)", xlabel="hours since first observation", ylabel="%")
    a.legend(fontsize=8, title=None)
fig.tight_layout()
save("eda_07_segment_timeseries")


# How hard is each segment to forecast with persistence? Normalised MAE at a 1 h horizon (train only).
def name(series, h=12):
    """Compute the Normalized Mean Absolute Error (NAME) for a given series and horizon."""
    s = np.asarray(series, float)
    return np.abs(s[h:] - s[:-h]).mean() / s.mean()


max_perc = 100
rows = [
    {"segment": f"{c}={k}", "VMs": int(tables[c].loc[k, "VMs"]), "NMAE@1h": name(g.to_numpy())}
    for c in tables
    for k, g in train_seg[train_seg[c].isin(tables[c].index[tables[c]["VMs"] >= max_perc])]
    .groupby(["timestamp", c])["avg_cpu"]
    .mean()
    .unstack()
    .items()
]
seg_pred = pd.DataFrame(rows).sort_values("NMAE@1h", ascending=False)
print(f"aggregate NMAE@1h = {name(s_tr):.4f}")
ipy_display(seg_pred.round(4))

**Operational reading.** Segments that carry a large share of VMs but low CPU (or the reverse) matter differently for capacity.
Smaller segments are noisier in relative terms (higher NAME) because they average over fewer VMs, so a segment-level policy needs wider
margins than the fleet-level one. Whether to forecast by segment is a trade-off between that extra noise and the ability to act per workload class.

## 6. Peak and low-demand periods: who drives them?

Peaks are defined on the training window only (aggregate at or above its 90th percentile). We then ask which segments move the aggregate.

In [ ]:
peak_ts = set(peaks["timestamp"])
rest_ts = set(tr["timestamp"]) - peak_ts
rows = []
max_perc = 100
for c in tables:
    big = tables[c].index[tables[c]["VMs"] >= max_perc]
    ts = train_seg[train_seg[c].isin(big)].groupby(["timestamp", c])["avg_cpu"].mean().unstack()
    for k in ts.columns:
        w = tables[c].loc[k, "vm_share"]
        d = ts.loc[ts.index.isin(peak_ts), k].mean() - ts.loc[ts.index.isin(rest_ts), k].mean()
        rows.append(
            {
                "segment": f"{c}={k}",
                "vm_share": w,
                "peak_minus_rest_%pts": d,
                "weighted_contribution": w * d,
            }
        )
contrib = pd.DataFrame(rows).sort_values("weighted_contribution", ascending=False)
ipy_display(contrib.round(4))
print(
    f"aggregate: peak steps {tr.loc[tr.timestamp.isin(peak_ts), col].mean():.3f} % vs rest {tr.loc[tr.timestamp.isin(rest_ts), col].mean():.3f} %"
)

**Operational reading.** The `weighted_contribution` column shows which segment explains the peak. Contributions that sum close to the aggregate
difference mean the peak is a broad, fleet-wide shift. A single dominant segment would instead justify a segment-specific trigger.

## 7. Why is the aggregate so smooth?

The mean of 10,000 VMs is far less volatile than a typical VM. How much smoother depends on how correlated VMs are. We back out the implied
average pairwise correlation from the variance of a mean (train only): `var(agg) = (sigma^2 / N) * (1 + (N - 1) * rho)`.

In [ ]:
tr_cpu = cpu[cpu["timestamp"] < CUT]
vm_var = tr_cpu.groupby("vm_id")["avg_cpu"].var(ddof=1)
vm_cv = (
    tr_cpu.groupby("vm_id")["avg_cpu"].std() / tr_cpu.groupby("vm_id")["avg_cpu"].mean()
).replace([np.inf, -np.inf], np.nan)
N = len(vm_var)
agg_var = float(np.var(s_tr, ddof=1))
sigma2 = float(vm_var.mean())
rho = (agg_var * N / sigma2 - 1) / (N - 1)
smooth = pd.Series(
    {
        "VMs (N)": N,
        "mean per-VM variance": sigma2,
        "aggregate variance": agg_var,
        "implied avg pairwise correlation": rho,
        "median per-VM CV (std/mean)": float(vm_cv.median()),
        "aggregate CV": float(np.std(s_tr, ddof=1) / np.mean(s_tr)),
    }
)
ipy_display(smooth.to_frame("value").round(5))

**Operational reading.** A small positive correlation is enough to leave a visible common movement in the aggregate even with 10,000 VMs. That common
component, not individual VM noise, is what a fleet-level forecast can learn. It is also why a forecast that is accurate to a fraction of a
percentage point can still be *operationally irrelevant*: the headroom question is dominated by how the busiest VMs and segments behave, not the fleet mean.

## 8. Forecast diagnostics (reads the pipeline outputs; holdout used for reporting only)

Model selection was finished on rolling-origin cross-validation inside the training window. Here we only look at what was selected and how it behaved.

In [ ]:
need = [
    TRAIN_DIR / "holdout_report.json",
    TRAIN_DIR / "cv_scores.csv",
    TRAIN_DIR / "intervals.csv",
    ROOT / "predictions" / "predictions.csv",
]
missing = [str(p) for p in need if not p.exists()]
if missing:
    raise FileNotFoundError(
        f"Run the pipeline first (uv run workload-forecasting predict). Missing: {missing}"
    )

rep = json.loads((TRAIN_DIR / "holdout_report.json").read_bytes())
cv = pd.read_csv(TRAIN_DIR / "cv_scores.csv")
pred = pd.read_csv(ROOT / "predictions" / "predictions.csv")
iv = pd.read_csv(TRAIN_DIR / "intervals.csv", header=[0, 1], index_col=0)
sel = rep["selected"]
print("selected models:", sel)

cv_cpu = cv[cv["target"] == col].set_index("model")
ipy_display(cv_cpu.round(4))

h_cpu = rep["holdout"][col]
ipy_display(
    pd.Series({k: v for k, v in h_cpu.items() if not isinstance(v, dict)})
    .to_frame("holdout")
    .round(4)
)
ipy_display(
    pd.DataFrame(
        {
            "point forecast": h_cpu["policy_point"],
            "upper band": h_cpu["policy_upper_band"],
        }
    ).round(4)
)

In [ ]:
lo = iv[("lower", col)].to_numpy()
hi = iv[("upper", col)].to_numpy()
p = pred[col].to_numpy()
a = test[col].to_numpy()
err = a - p

fig, ax = plt.subplots(1, 2, figsize=(15, 4.2))
tail = tr.iloc[-48:]
ax[0].plot(hours(tail.timestamp), tail[col], color="C0", label="train (last 4 h)")
ax[0].plot(hours(test.timestamp), a, color="k", label="holdout actual")
ax[0].plot(hours(test.timestamp), p, color="C3", label=f"forecast ({sel[col]})")
ax[0].fill_between(
    hours(test.timestamp), lo, hi, color="C3", alpha=0.2, label="90% empirical CV band"
)
ax[0].set(title="Holdout forecast", xlabel="hours since first observation", ylabel="%")
ax[0].legend(fontsize=8)

order = cv_cpu.sort_values("mae")
colors = ["C3" if m == sel[col] else "C0" for m in order.index]
ax[1].barh(order.index, order["mae"], color=colors)
ax[1].invert_yaxis()
ax[1].set(title="Cross-validation MAE (train only); selected in red", xlabel="MAE (% points)")
fig.tight_layout()
save("eda_08_forecast")

the = tr[col].quantile(0.9)
peak_mask = a >= the
by_step = pd.DataFrame(
    {
        "step": np.arange(1, H + 1),
        "actual": a,
        "forecast": p,
        "error": err,
        "abs_error": np.abs(err),
    }
)
print(
    f"bias (mean error) = {err.mean():+.4f} | share of steps actual above forecast = {(err > 0).mean():.0%}"
)
print(
    f"abs error, first 6 steps {np.abs(err[:6]).mean():.4f} vs last 6 steps {np.abs(err[-6:]).mean():.4f}"
)
print(
    f"holdout steps at/above the train p90 ({the:.3f}): {int(peak_mask.sum())}"
    + (
        f" -> MAE there {np.abs(err[peak_mask]).mean():.4f}"
        if peak_mask.any()
        else " (no peak occurs in the holdout, so peak error cannot be assessed here)"
    )
)
print(
    f"cross-validation peak MAE of the selected model {cv_cpu.loc[sel[col], 'peak_mae']:.4f} vs Naive {cv_cpu.loc['Naive', 'peak_mae']:.4f}"
)
ipy_display(by_step.round(3).head(8))

**Operational reading.** Look at three things: (1) whether the error is a persistent bias (the level shifted) or noise around the forecast,
(2) whether error grows with the horizon, and (3) whether the selected model is worse than the baseline exactly when load is high,
which is when under-provisioning hurts. The cross-validation peak MAE line answers (3) using only training data.

## 9. Export the numbers used in the report

In [ ]:
def _j(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    return str(o)


summary = {
    "data_quality": quality["value"].to_dict(),
    "lifetime_status": status.to_dict(),
    "median_lifetime_h": float(life.median()),
    "cpu_stats": stats,
    "adf_p": {"level": adf_level, "first_difference": adf_diff},
    "train_range_pct": [float(s_tr.min()), float(s_tr.max())],
    "peak_threshold_pct": float(hi_the),
    "persistence_vs_mean": horizon.reset_index().to_dict(orient="records"),
    "implied_pairwise_corr": float(rho),
    "aggregate_name_1h": float(name(s_tr)),
    "segments": {c: t.round(4).reset_index().to_dict(orient="records") for c, t in tables.items()},
    "peak_contrib": contrib.round(4).to_dict(orient="records"),
}
(ROOT / "reports" / "eda_summary.json").write_text(json.dumps(summary, indent=2, default=_j))
print("wrote reports/eda_summary.json")